# T34 - 债券新发行 重构Notebook

## 项目概述

债券新发行系统是一个综合性的债券一级市场数据采集和分析工具，主要功能包括：

1. **债券到期数据采集**：从Wind API获取各种类型债券的发行和到期数据
2. **债券新发行分析**：从同花顺iFinD API获取新发债券的详细信息
3. **数据存储与管理**：将数据存储到MySQL数据库，支持动态列管理
4. **数据去重与清洗**：自动识别和删除重复数据

---

## 1. 环境配置

导入必要的依赖包和配置参数。

In [ ]:
# 1.1 导入标准库
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime, timedelta

# 1.2 导入数据处理库
import pandas as pd
import numpy as np

# 1.3 导入数据库相关库
import sqlalchemy
from sqlalchemy import inspect, text
import pymysql

# 1.4 导入配置
from config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR,
    MYSQL_HOST, MYSQL_PORT, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DATABASE,
    BOND_TYPES, IFIND_BOND_TYPES,
    BOND_MATURITY_CONFIG, BOND_ISSUE_CONFIG,
    get_mysql_connection_string, print_config_summary
)

# 1.5 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('T34_债券新发行')

# 1.6 打印配置摘要
print_config_summary()

In [ ]:
# 1.7 创建数据库连接
def create_db_engine():
    connection_string = get_mysql_connection_string()
    engine = sqlalchemy.create_engine(
        connection_string,
        poolclass=sqlalchemy.pool.NullPool,
        pool_recycle=3600
    )
    return engine

sql_engine = create_db_engine()
print(f'数据库连接创建成功: {MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}')

---

## 2. 工具函数

定义可复用的工具函数。

In [ ]:
# 2.1 列名映射函数
def generate_column_mapping(columns):
    """生成列名映射字典"""
    return {col: f'col_{i}' for i, col in enumerate(columns)}


# 2.2 列名转换函数
def change_column_names(engine, table_name, column_mapping, to_english=True):
    """修改数据库表的列名"""
    with engine.begin() as connection:
        for original_name, new_name in column_mapping.items():
            if to_english:
                sql = f'ALTER TABLE `{table_name}` CHANGE `{original_name}` `{new_name}` VARCHAR(255);'
            else:
                sql = f'ALTER TABLE `{table_name}` CHANGE `{new_name}` `{original_name}` VARCHAR(255);'
            connection.execute(text(sql))

print('列名映射函数定义完成')

In [ ]:
# 2.3 数据处理与入库函数
def pro_data(df, table_name, engine):
    """处理数据并插入数据库"""
    wsd_data = df.copy()
    
    if 'dt' in wsd_data.columns:
        wsd_data['dt'] = pd.to_datetime(wsd_data['dt'], errors='coerce')
        wsd_data = wsd_data.dropna(subset=['dt'])
        wsd_data['dt'] = wsd_data['dt'].dt.strftime('%Y-%m-%d')
    
    wsd_data = wsd_data.replace({pd.NA: None, pd.NaT: None})
    wsd_data = wsd_data.where(pd.notnull(wsd_data), None)
    
    if wsd_data.empty:
        print('No valid data to insert.')
        return 0
    
    column_mapping = generate_column_mapping(wsd_data.columns)
    wsd_data = wsd_data.rename(columns=column_mapping)
    
    inspector = inspect(engine)
    table_exists = inspector.has_table(table_name)
    
    if table_exists:
        existing_columns = inspector.get_columns(table_name)
        existing_columns_names = [col['name'] for col in existing_columns]
        wsd_data_columns = wsd_data.columns.tolist()
        
        for col in wsd_data_columns:
            if col not in existing_columns_names:
                col_type = 'FLOAT' if pd.api.types.is_numeric_dtype(wsd_data[col]) else 'VARCHAR(255)'
                with engine.begin() as connection:
                    connection.execute(text(f'ALTER TABLE `{table_name}` ADD COLUMN `{col}` {col_type};'))
        
        insert_columns = ', '.join([f'`{col}`' for col in wsd_data_columns])
        update_columns = ', '.join([f'`{col}` = VALUES(`{col}`)' for col in wsd_data_columns])
        value_placeholders = ', '.join([f':{col}' for col in wsd_data_columns])
        
        insert_query = text(f'''INSERT INTO `{table_name}` ({insert_columns}) VALUES ({value_placeholders}) ON DUPLICATE KEY UPDATE {update_columns};''')
        
        inserted_count = 0
        with engine.begin() as connection:
            for _, row in wsd_data.iterrows():
                try:
                    params = {col: row[col] for col in wsd_data_columns}
                    connection.execute(insert_query, params)
                    inserted_count += 1
                except Exception as e:
                    print(f'Error inserting row: {e}')
        
        print(f'数据插入完成: {table_name}, 共 {inserted_count} 条')
        return inserted_count
    return 0

print('数据处理函数定义完成')

In [ ]:
# 2.4 数据去重函数
def deduplicate_bond_data(engine, table_name):
    """债券新发行数据去重"""
    with engine.begin() as connection:
        connection.execute(text(f'''CREATE TEMPORARY TABLE temp_table AS SELECT * FROM `{table_name}` WHERE sec_name IN (SELECT sec_name FROM `{table_name}` GROUP BY sec_name HAVING COUNT(*) = 1) UNION SELECT * FROM `{table_name}` WHERE trade_code IN (SELECT MIN(trade_code) FROM `{table_name}` WHERE trade_code NOT LIKE 'S%%' GROUP BY sec_name HAVING COUNT(*) > 1);'''))
        connection.execute(text(f'DELETE FROM `{table_name}`;'))
        connection.execute(text(f'INSERT INTO `{table_name}` SELECT * FROM temp_table;'))
        connection.execute(text('DROP TEMPORARY TABLE temp_table;'))
    print(f'数据去重完成: {table_name}')

print('数据去重函数定义完成')

---

## 3. 数据获取

从Wind API和同花顺iFinD API获取债券数据。

In [ ]:
# 3.1 债券到期数据采集 (Wind API)
def fetch_bond_maturity_data(engine):
    """从Wind API获取债券到期数据 - 详细实现见 src/zqdq.py"""
    print('债券到期数据采集函数 - 需要Wind API')

print('债券到期数据采集函数定义完成')

In [ ]:
# 3.2 债券新发行数据采集 (iFinD API)
def fetch_bond_new_issue_data(engine):
    """从同花顺iFinD API获取债券新发行数据 - 详细实现见 src/zqfx.py"""
    print('债券新发行数据采集函数 - 需要iFinD API')

print('债券新发行数据采集函数定义完成')

---

## 4. 核心逻辑

主函数实现和执行入口。

In [ ]:
# 5.1 主执行函数
def main():
    print('='*60)
    print('T34 - 债券新发行数据采集系统')
    print('='*60)
    print(f'执行时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    
    engine = create_db_engine()
    
    print('\n' + '='*60)
    print('Step 1: 债券到期数据采集 (Wind API)')
    print('='*60)
    fetch_bond_maturity_data(engine)
    
    print('\n' + '='*60)
    print('Step 2: 债券新发行数据采集 (iFinD API)')
    print('='*60)
    fetch_bond_new_issue_data(engine)
    
    print('\n' + '='*60)
    print('执行完成')
    print('='*60)

print('主执行函数定义完成')

---

## 5. 执行与测试

执行主流程并验证结果。

In [ ]:
# 6.1 执行主函数
# 取消下面的注释来执行
# main()

In [ ]:
# 6.2 测试数据库连接
def test_database_connection():
    try:
        with sql_engine.begin() as connection:
            result = connection.execute(text('SELECT 1')).scalar()
            print(f'数据库连接测试: {"成功" if result == 1 else "失败"}')
            return True
    except Exception as e:
        print(f'数据库连接测试失败: {e}')
        return False

test_database_connection()

---

## 6. 工具函数汇总

| 函数名 | 功能描述 |
|--------|----------|
| generate_column_mapping | 生成列名映射字典 |
| change_column_names | 修改数据库表列名 |
| pro_data | 数据处理与入库 |
| deduplicate_bond_data | 债券数据去重 |
| fetch_bond_maturity_data | 债券到期数据采集 |
| fetch_bond_new_issue_data | 债券新发行数据采集 |
| main | 主执行函数 |